# Your first penalized trend

We separate noisy observations into a fitted **in-sample trend** and residuals. The finite-difference penalty controls roughness, not predictive ability. Our synthetic truth is known so we can evaluate recovery transparently.

## A reproducible signal

Construct a linear drift plus a slowly varying periodic component and white noise. A fixed seed makes every figure reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import TrendSmoother, plot_trend
rng=np.random.default_rng(21)
t=np.arange(100); truth=.025*t+.5*np.sin(t/19)
y=truth+rng.normal(0,.38,len(t))
fig,ax=plt.subplots(figsize=(10,3.5))
ax.plot(t,y,color="0.7",label="observed"); ax.plot(t,truth,label="latent trend")
ax.set(title="Observed series and the simulated truth",xlabel="Time"); ax.legend(); plt.show()

## Compare penalties

At λ=0 the trend interpolates the data; larger penalties suppress finite differences. The operation is *two-sided smoothing*: it uses the complete sample.

In [ ]:
solver=TrendSmoother(len(y),order=2)
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(y,color="0.8",label="observed")
ax.plot(truth,color="k",ls=":",label="true trend")
for lam in [0,2,50,1000]:
    ax.plot(solver.fit(y,lambda_=lam).trend,label=f"λ={lam}")
ax.set(title="One difference order, several smoothing strengths",xlabel="Index")
ax.legend(ncol=3);plt.show()

## Residuals and roughness

The fit–smoothness trade-off can be examined through residual RMS error and the norm of second differences; neither metric alone chooses a uniquely correct trend.

In [ ]:
lambdas=np.geomspace(.01,1e4,60)
residual_rms=[];roughness=[]
for lam in lambdas:
    f=solver.fit(y,lambda_=float(lam))
    residual_rms.append(float(np.sqrt(np.mean(f.residual**2))))
    roughness.append(float(np.linalg.norm(solver.D@f.trend)))
fig,ax=plt.subplots(1,2,figsize=(11,3.5))
ax[0].semilogx(lambdas,residual_rms);ax[0].set(xlabel="λ",ylabel="Residual RMSE")
ax[1].loglog(lambdas,roughness);ax[1].set(xlabel="λ",ylabel="||D₂ trend||")
fig.tight_layout();plt.show()
print("The residual RMSE generally rises as roughness decreases.")